# Notebook 01 — Circuit Building

**Predicting the Success Probability of Deutsch-Jozsa, Bernstein-Vazirani, and Simon's Algorithms Under Noise Using Classical Machine Learning**

---

## Objective
In this notebook, we build and visually inspect the quantum circuits for three foundational oracle-based algorithms:

1. **Deutsch-Jozsa (DJ)** — Determines if a function is constant or balanced
2. **Bernstein-Vazirani (BV)** — Recovers a hidden bitstring
3. **Simon's Algorithm** — Finds the hidden period of a 2-to-1 mapping

All three algorithms are **deterministic** in an ideal noiseless setting — they succeed with certainty on a single run. This property makes them ideal testbeds for isolating the effect of noise.

### What we'll do:
- Build oracle circuits for each algorithm
- Draw and inspect the full quantum circuits
- Verify correctness by running on an ideal (noiseless) simulator
- Tabulate circuit statistics (depth, gate counts) across qubit counts 2–8

## 1.1 Imports and Setup

In [ ]:
import sys
import os

# Add project root to Python path so we can import from src/
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
from qiskit import transpile
from qiskit_aer import AerSimulator

# Our project modules
from src.circuits import (
    build_dj_oracle,
    build_dj_circuit,
    build_bv_circuit,
    build_simon_circuit,
    check_simon_success,
    generate_secret_strings,
    generate_all_circuit_configs,
    get_circuit_stats,
)

print('All imports successful!')
print(f'Project root: {project_root}')

### Helper Function
A utility to run a circuit on the ideal (noiseless) Aer simulator and return measurement counts.

In [ ]:
def run_ideal(circuit, shots=4096):
    """Run a circuit on the ideal Aer simulator."""
    sim = AerSimulator()
    transpiled = transpile(circuit, sim)
    result = sim.run(transpiled, shots=shots).result()
    return result.get_counts()

---

## 1.2 Deutsch-Jozsa Algorithm

### How it works:
The Deutsch-Jozsa algorithm determines whether a black-box function $f: \{0,1\}^n \rightarrow \{0,1\}$ is **constant** (same output for all inputs) or **balanced** (outputs 0 for exactly half, 1 for the other half).

**Circuit steps:**
1. Initialize ancilla qubit to $|1\rangle$
2. Apply Hadamard gates to all qubits
3. Apply the oracle $U_f$
4. Apply Hadamard gates to input qubits
5. Measure input qubits

**Expected output:**
- Constant oracle → measure $|00\ldots0\rangle$
- Balanced oracle → measure any non-zero bitstring

### 1.2.1 Constant Oracle (f(x) = 0 for all x)

In [ ]:
# Build a constant-0 oracle for 3 qubits
n = 3
oracle_const = build_dj_oracle(n, 'constant_0')
circuit_const = build_dj_circuit(n, oracle_const)

print(f'DJ Circuit (Constant-0, n={n}):')
print(f'Total qubits: {circuit_const.num_qubits} ({n} input + 1 ancilla)')
print(f'Classical bits: {circuit_const.num_clbits}')
print()
circuit_const.draw(output='mpl', style='iqp', fold=40)

In [ ]:
# Verify: should output '000' with 100% probability
counts = run_ideal(circuit_const)
print(f'Measurement results: {counts}')
print(f'Expected: {{"000": 4096}}')
print(f'✓ PASS' if counts.get('000', 0) == 4096 else '✗ FAIL')

### 1.2.2 Balanced Oracle

In [ ]:
# Build a balanced oracle for 3 qubits
oracle_bal = build_dj_oracle(n, 'balanced', seed=42)
circuit_bal = build_dj_circuit(n, oracle_bal)

print(f'DJ Circuit (Balanced, n={n}):')
circuit_bal.draw(output='mpl', style='iqp', fold=40)

In [ ]:
# Verify: should NOT output '000'
counts = run_ideal(circuit_bal)
print(f'Measurement results: {counts}')
print(f'Expected: any non-zero bitstring (NOT "000")')
print(f'✓ PASS' if '000' not in counts else '✗ FAIL')

---

## 1.3 Bernstein-Vazirani Algorithm

### How it works:
The Bernstein-Vazirani algorithm recovers a hidden bitstring $s \in \{0,1\}^n$ from an oracle that computes $f(x) = s \cdot x \pmod{2}$ (bitwise inner product).

**Circuit steps:**
1. Initialize ancilla to $|1\rangle$
2. Apply Hadamard gates to all qubits
3. Apply the oracle (CNOT from each qubit where $s_i = 1$ to the ancilla)
4. Apply Hadamard gates to input qubits
5. Measure — the result IS the secret string $s$

**Key property:** BV recovers $s$ in a **single query**, compared to $n$ classical queries.

In [ ]:
# Build BV circuit with secret string '101'
secret = '101'
n = len(secret)
circuit_bv = build_bv_circuit(n, secret)

print(f'BV Circuit (secret="{secret}", n={n}):')
print(f'Total qubits: {circuit_bv.num_qubits} ({n} input + 1 ancilla)')
print()
circuit_bv.draw(output='mpl', style='iqp', fold=40)

In [ ]:
# Verify: should recover the secret string '101' with 100% probability
counts = run_ideal(circuit_bv)
print(f'Measurement results: {counts}')
print(f'Expected: {{"{secret}": 4096}}')
print(f'✓ PASS' if counts.get(secret, 0) == 4096 else '✗ FAIL')

In [ ]:
# Test with multiple secret strings
print('Testing BV with multiple secret strings:\n')
test_secrets = ['10', '110', '1010', '11011']

for secret in test_secrets:
    n = len(secret)
    circuit = build_bv_circuit(n, secret)
    counts = run_ideal(circuit)
    recovered = max(counts, key=counts.get)
    status = '✓' if recovered == secret else '✗'
    print(f'  {status} secret="{secret}" → recovered="{recovered}" '
          f'(probability: {counts[recovered]/4096:.2%})')

---

## 1.4 Simon's Algorithm

### How it works:
Simon's algorithm finds a hidden bitstring $s$ such that for a function $f$:
$$f(x) = f(y) \iff x \oplus y \in \{0^n, s\}$$

This means $f$ is either 1-to-1 (if $s = 0^n$) or 2-to-1.

**Circuit steps:**
1. Apply Hadamard to input register
2. Apply the oracle
3. Apply Hadamard to input register
4. Measure input register

Each measurement produces a bitstring $z$ satisfying $z \cdot s = 0 \pmod{2}$. Multiple runs + classical post-processing determine $s$.

**Key property:** Historically significant as the precursor to Shor's algorithm.

In [ ]:
# Build Simon's circuit with secret period '10'
secret = '10'
n = len(secret)
circuit_simon = build_simon_circuit(n, secret)

print(f"Simon's Circuit (period=\"{secret}\", n={n}):")
print(f'Total qubits: {circuit_simon.num_qubits} ({n} input + {n} output)')
print()
circuit_simon.draw(output='mpl', style='iqp', fold=40)

In [ ]:
# Verify: all outputs z should satisfy z · s = 0 (mod 2)
counts = run_ideal(circuit_simon)
print(f'Measurement results: {counts}')
print(f'Hidden period s = "{secret}"')
print()

s_bits = [int(b) for b in reversed(secret)]
print('Checking z · s = 0 (mod 2) for each output:')
for bitstring, count in sorted(counts.items(), key=lambda x: -x[1]):
    z_bits = [int(b) for b in reversed(bitstring)]
    dot = sum(z * s for z, s in zip(z_bits, s_bits)) % 2
    status = '✓' if dot == 0 else '✗'
    print(f'  {status} z={bitstring}, z·s = {dot} (count: {count})')

success = check_simon_success(counts, secret, n)
print(f'\nOverall success rate: {success:.2%}')

---

## 1.5 Correctness Verification — All Algorithms

Let's systematically verify that all circuits produce correct outputs under ideal (noiseless) conditions across multiple qubit counts.

In [ ]:
from src.simulation import run_circuit, compute_success_probability

print('CORRECTNESS VERIFICATION — Noiseless Simulation')
print('=' * 65)

configs = generate_all_circuit_configs(range(2, 6))  # 2 to 5 qubits for speed

results = []
for cfg in configs:
    counts = run_circuit(cfg['circuit'], noise_model=None, shots=4096)
    success = compute_success_probability(
        counts, cfg['expected_output'], cfg['algorithm'],
        n_qubits=cfg['n_qubits'], secret_string=cfg.get('expected_output')
    )
    results.append({
        'algorithm': cfg['algorithm'],
        'n_qubits': cfg['n_qubits'],
        'oracle': cfg['oracle_desc'],
        'success': success,
    })

df_verify = pd.DataFrame(results)
print(f'\nTotal circuits tested: {len(df_verify)}')
print(f'All passed (success ≥ 0.99): {(df_verify["success"] >= 0.99).all()}')
print(f'\nSuccess probability statistics:')
print(df_verify.groupby('algorithm')['success'].describe().round(4))

---

## 1.6 Circuit Statistics

Here we tabulate the **transpiled circuit depth** and **gate counts** for each algorithm across qubit counts 2–8. These will be important features in our ML model.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')

# Collect circuit stats across all configs
configs_full = generate_all_circuit_configs(range(2, 9))
stats_data = []

for cfg in configs_full:
    stats = get_circuit_stats(cfg['circuit'])
    stats['algorithm'] = cfg['algorithm']
    stats['n_qubits'] = cfg['n_qubits']
    stats_data.append(stats)

df_stats = pd.DataFrame(stats_data)

print('Circuit Statistics (Transpiled):')
print('=' * 60)
print(df_stats.groupby(['algorithm', 'n_qubits'])[['circuit_depth', 'gate_count_2q', 'total_gate_count']]
      .mean().round(1))

In [ ]:
# Plot: Circuit Depth vs Qubit Count by Algorithm
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for algo in df_stats['algorithm'].unique():
    subset = df_stats[df_stats['algorithm'] == algo]
    grouped = subset.groupby('n_qubits')['circuit_depth'].mean()
    label = algo.replace('_', ' ').title()
    axes[0].plot(grouped.index, grouped.values, marker='o', linewidth=2,
                 markersize=7, label=label)
    
    grouped_2q = subset.groupby('n_qubits')['gate_count_2q'].mean()
    axes[1].plot(grouped_2q.index, grouped_2q.values, marker='s', linewidth=2,
                 markersize=7, label=label)

axes[0].set_xlabel('Number of Qubits', fontsize=12)
axes[0].set_ylabel('Circuit Depth', fontsize=12)
axes[0].set_title('Circuit Depth vs Qubit Count', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

axes[1].set_xlabel('Number of Qubits', fontsize=12)
axes[1].set_ylabel('Two-Qubit Gate Count', fontsize=12)
axes[1].set_title('Two-Qubit Gates vs Qubit Count', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Transpiled Circuit Complexity', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/circuit_complexity.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figure saved to outputs/figures/circuit_complexity.png')

---

## Summary

In this notebook we:

1. **Built** oracle circuits for Deutsch-Jozsa (constant + balanced), Bernstein-Vazirani, and Simon's algorithms
2. **Drew** the full quantum circuits to visually inspect their structure
3. **Verified** correctness: all circuits produce the correct output with 100% probability under noiseless conditions
4. **Tabulated** transpiled circuit statistics (depth, gate counts) across qubit counts 2–8

**Key observation:** Circuit depth and two-qubit gate count grow with qubit count, and Simon's algorithm uses significantly more qubits (2n) than DJ or BV (n+1). These structural differences will affect how each algorithm responds to noise.

**Next:** In Notebook 02, we'll add noise to these circuits and run a full parameter sweep.